# CASMI26: loading and scaling benchmarks

**Strand:** data and validation (Luke Devlin). **Feeds:** the pipeline part of
`report/04-Scaling` and the 1000× argument for the data stage.

Notebook 01 cleaned the 2.5 million training spectra in about a minute on a laptop. This notebook
asks what that minute is made of and how it would grow. It answers three questions by
measurement:

1. **Loading.** How do time and memory grow with the amount of data read, for four ways of
   reading the same Parquet file?
2. **Data volume.** Does the cleaning pass grow linearly with the number of spectra, and does
   its memory stay flat? Linearity is what makes the 1000× extrapolation legitimate.
3. **Cores.** How much faster is the cleaning pass with more cores, and how do four
   implementations compare: one process, Polars' own threads, `joblib` processes (as taught in
   Block 09) and PySpark (Block 11)? Amdahl's law is fitted to each.

Section 7 turns the measurements into an estimate of what cleaning 1000 times the data would
take, with every assumption stated.

**Before running.** Run notebook 01 first: this notebook reads the same `train.parquet`, and
checks its counts against notebook 01's `manifest.json`. PySpark needs a Java runtime, version 17,
21 or 25 (section 1.2 checks this and explains what to install). Without Java, the Spark runs are
skipped and everything else still runs.

**Run time.** The full set of runs takes about half an hour on an 8-core laptop, and uses up to
about 4 GB of memory at 8 workers. Set `QUICK = True` in the settings cell to run every
experiment on 4 row groups, once each, in a few minutes. Set `RUN_BENCHMARKS = False` to skip the
runs and analyse the results saved by an earlier run.

## 0. Setup

In [ ]:
# Install anything that is missing into the environment this kernel runs in.
import importlib.util
import subprocess
import sys

REQUIRED = {"polars": "polars", "pyarrow": "pyarrow", "numpy": "numpy", "matplotlib": "matplotlib",
            "pandas": "pandas", "joblib": "joblib", "psutil": "psutil", "pyspark": "pyspark"}
missing = [package for module, package in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("installed:" if missing else "all packages already present", *missing)

In [ ]:
import json
import os
import platform
import re
import shutil
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import psutil
import pyarrow as pa
import pyarrow.parquet as pq

print("python", sys.version.split()[0], "| polars", pl.__version__, "| pyarrow", pa.__version__,
      "| pandas", pd.__version__, "| joblib", joblib.__version__)

Every setting that changes a result lives in the next cell.

In [ ]:
# ---- Switches ------------------------------------------------------------------------------------
QUICK = False            # True: 4 row groups and one repeat per run, a few minutes in all
RUN_BENCHMARKS = True    # False: skip the runs and analyse the saved results (section 2.3)
RUN_SPARK = True         # the Spark runs need Java (section 1.2); they are skipped without it

# ---- Experiment design ---------------------------------------------------------------------------
REPEATS = 3                          # runs of every configuration; the median is reported
WORKER_COUNTS = [1, 2, 4, 6, 8]      # cores to try; capped at this machine's logical cores
SIZE_STEPS = [1, 2, 5, 10, None]     # row groups read in sections 3 and 4; None means all of them
MEMORY_BUDGET_GB = None              # in-memory loads predicted to need more than this are skipped;
                                     # None means half of this machine's memory
SPARK_DRIVER_MEMORY = "2g"

# ---- Cleaning: the same values as notebook 01, section 5 ------------------------------------------
CLEANING = {
    "MIN_REL_INTENSITY": 0.005, "MAX_PEAKS": 256, "PRECURSOR_WINDOW": 2.5,
    "PRECURSOR_MZ_RANGE": [50.0, 1500.0], "MIN_PEAKS_TRAIN": 3, "BIN_WIDTH": 0.1, "BATCH_ROWS": 32_768,
}

# ---- The 1000× projection (section 7). These are assumptions, not measurements. --------------------
SCALE = 1000
CLUSTER_MACHINES = 50                # machines like this laptop in the hypothetical cluster
NETWORK_GBIT_PER_S = 10              # network bandwidth per machine for the shuffle in the reduce
DISK_MB_PER_S = 1000                 # read speed of each machine's disk (a fast NVMe drive is 2,000 to 7,000)

if QUICK:
    REPEATS, SIZE_STEPS = 1, [1, 2, 4]

In [ ]:
# One plot style for the whole notebook, the same as notebook 01.
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_thousands_separator(",")
pl.Config.set_float_precision(3)

INK, INK_SOFT, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
BLUE, ORANGE, AQUA, PURPLE, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#7a5cd6", "#898781"
# One colour per implementation or loading method, used the same way in every figure.
IMPLEMENTATIONS = {"one process": GREY, "polars threads": AQUA, "joblib": BLUE, "spark": ORANGE}
LOADERS = {"pandas": ORANGE, "polars": AQUA, "polars streaming": BLUE, "pyarrow batches": PURPLE}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "font.size": 10,
    "axes.edgecolor": "#c3c2b7", "axes.linewidth": 0.8, "axes.labelcolor": INK_SOFT,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlecolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK_SOFT, "ytick.labelcolor": INK_SOFT,
    "legend.frameon": False, "lines.linewidth": 2,
})

## 1. The machine and the data

### 1.1 This machine

Benchmarks only mean something next to the hardware they ran on, so the notebook records it. A
*logical* core is what the operating system schedules on. With hyper-threading, two logical cores
share one *physical* core, and a CPU-bound job gains little from the second one.

In [ ]:
COMPETITION = "enveda-CASMI26-molecule-id-mass-spectra"
HERE = Path.cwd()


def find_repo_root(start: Path) -> Path:
    """The first folder at or above `start` that holds the git repo or the data."""
    for folder in [start, *start.parents]:
        if (folder / ".git").exists() or (folder / COMPETITION).exists():
            return folder
    return start


def find_data_dir(repo: Path) -> Path:
    candidates = [os.environ.get("CASMI26_DATA_DIR"), repo / COMPETITION, Path("/kaggle/input") / COMPETITION]
    for candidate in candidates:
        if candidate and (Path(candidate) / "train.parquet").exists():
            return Path(candidate)
    raise FileNotFoundError("train.parquet not found: run notebook 01 first, which downloads it")


REPO = find_repo_root(HERE)
TRAIN = find_data_dir(REPO) / "train.parquet"
MANIFEST = REPO / f"{COMPETITION}-derived" / "manifest.json"
SCRATCH = REPO / f"{COMPETITION}-benchmark-scratch"   # cleaned output of each run; deleted after it
RESULTS_DIR = HERE / "benchmark_results" / ("quick" if QUICK else "full")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LOGICAL_CORES = os.cpu_count()
PHYSICAL_CORES = psutil.cpu_count(logical=False)
MEMORY_GB = psutil.virtual_memory().total / 1e9
WORKER_COUNTS = [p for p in WORKER_COUNTS if p <= LOGICAL_CORES]
if MEMORY_BUDGET_GB is None:
    MEMORY_BUDGET_GB = MEMORY_GB / 2

machine = {
    "system": f"{platform.system()} {platform.release()}", "processor": platform.processor() or platform.machine(),
    "logical cores": LOGICAL_CORES, "physical cores": PHYSICAL_CORES, "memory (GB)": round(MEMORY_GB, 1),
    "python": sys.version.split()[0], "polars": pl.__version__, "pyarrow": pa.__version__,
    "pandas": pd.__version__, "joblib": joblib.__version__,
}
pl.DataFrame({"": list(machine), "value": [str(v) for v in machine.values()]})

### 1.2 Java, for Spark

Spark is written in Scala and runs on the Java virtual machine (JVM). PySpark is a Python front
end: it starts a JVM, and the JVM starts Python worker processes to run our Python functions. So
PySpark needs a Java runtime. The [Spark documentation](https://spark.apache.org/docs/latest/)
lists Java 17, 21 and 25 as supported. The cell below finds Java, reports its version, and starts
a two-core Spark session to check that the pieces work together.

If Java is missing, install a Java 21 JDK and run the notebook again:

| System | Command |
| --- | --- |
| Ubuntu, Debian | `sudo apt install openjdk-21-jdk` |
| macOS (Homebrew) | `brew install openjdk@21`, then follow its instructions to link it |
| Windows | Install Temurin 21 from [adoptium.net](https://adoptium.net/) and tick "Set JAVA_HOME" |

If several versions are installed, point the `JAVA_HOME` environment variable at the one to use.

In [ ]:
def java_version() -> tuple[str | None, int | None]:
    """The java executable Spark will use, and its major version."""
    home = os.environ.get("JAVA_HOME")
    java = str(Path(home) / "bin" / "java") if home else shutil.which("java")
    if not java or not Path(java).exists() and not shutil.which(java):
        return None, None
    try:
        text = subprocess.run([java, "-version"], capture_output=True, text=True, timeout=30).stderr
    except (OSError, subprocess.SubprocessError):
        return java, None
    match = re.search(r'version "(\d+)(?:\.(\d+))?', text)
    if not match:
        return java, None
    major = int(match.group(1))
    return java, int(match.group(2)) if major == 1 else major   # Java 8 reports itself as 1.8


def worker_env(threads: int) -> dict:
    """The environment of a benchmark process: its Polars and Arrow thread count, and where to find
    casmi_bench.py and this Python, for the worker processes that joblib and Spark start."""
    return {**os.environ,
            "POLARS_MAX_THREADS": str(threads),
            "BENCH_THREADS": str(threads),
            "PYTHONPATH": os.pathsep.join(filter(None, [str(HERE), os.environ.get("PYTHONPATH")])),
            "PYSPARK_PYTHON": sys.executable, "PYSPARK_DRIVER_PYTHON": sys.executable}


JAVA, JAVA_MAJOR = java_version()
SUPPORTED_JAVA = (17, 21, 25)
if not RUN_SPARK:
    print("RUN_SPARK is off: the Spark runs are skipped")
elif JAVA is None:
    RUN_SPARK = False
    print("No Java runtime found, so the Spark runs are skipped. Install Java 21 (see the table above).")
else:
    note = "" if JAVA_MAJOR in SUPPORTED_JAVA else f" (not one of {SUPPORTED_JAVA}; trying it anyway)"
    print(f"Java {JAVA_MAJOR} at {JAVA}{note}")
    probe = ("from pyspark.sql import SparkSession\n"
             "s = SparkSession.builder.master('local[2]').config('spark.ui.enabled', 'false').getOrCreate()\n"
             "s.sparkContext.setLogLevel('ERROR')\n"
             "print('SPARK_OK', s.version, s.sparkContext.parallelize(range(10), 2).map(lambda x: x * x).sum())\n"
             "s.stop()\n")
    try:
        check = subprocess.run([sys.executable, "-c", probe], env=worker_env(1), capture_output=True,
                               text=True, timeout=300)
        ok = "SPARK_OK" in check.stdout
    except subprocess.TimeoutExpired:
        ok, check = False, None
    if ok:
        print("Spark works:", check.stdout.split("SPARK_OK", 1)[1].strip(), "(the version, and the sum of 0² to 9², which is 285)")
    else:
        RUN_SPARK = False
        print("Spark did not start, so the Spark runs are skipped. The last lines of its error:")
        print("\n".join((check.stderr if check else "timed out after 5 minutes").strip().splitlines()[-8:]))
        if JAVA_MAJOR not in SUPPORTED_JAVA:
            print(f"Java {JAVA_MAJOR} is the likely cause: install Java 21 and set JAVA_HOME to it.")

### 1.3 The data

The unit of work throughout is the Parquet *row group* (notebook 01, section 2.1): the file is
stored in blocks of rows that can be read independently. They are the natural tasks to hand to
parallel workers, and reading the first *k* of them gives nested subsets of the data for the
data-volume experiments.

In [ ]:
layout = pq.ParquetFile(TRAIN).metadata
N_ROW_GROUPS = layout.num_row_groups
ROW_GROUP_ROWS = np.array([layout.row_group(i).num_rows for i in range(N_ROW_GROUPS)])
ROW_GROUP_BYTES = np.array([layout.row_group(i).total_byte_size for i in range(N_ROW_GROUPS)])   # uncompressed
FILE_BYTES = TRAIN.stat().st_size
ALL_COLUMNS = pq.ParquetFile(TRAIN).schema_arrow.names
METADATA_COLUMNS = [c for c in ALL_COLUMNS if c not in ("ms2_mzs", "ms2_normalized_intensities", "collision_energy_ev")]

FULL = 4 if QUICK else N_ROW_GROUPS            # row groups used by the experiments of sections 4 and 5
SIZES = sorted({min(s or N_ROW_GROUPS, FULL) for s in SIZE_STEPS})
print(f"{TRAIN.name}: {layout.num_rows:,} spectra in {N_ROW_GROUPS} row groups, {FILE_BYTES / 1e9:.2f} GB on disk, "
      f"{ROW_GROUP_BYTES.sum() / 1e9:.2f} GB uncompressed")
print(f"row groups hold {ROW_GROUP_ROWS.min():,} to {ROW_GROUP_ROWS.max():,} spectra")
print(f"experiments use {FULL} row groups ({ROW_GROUP_ROWS[:FULL].sum():,} spectra); data-volume steps: {SIZES}")

## 2. The workload and how it is timed

### 2.1 The code that every implementation runs

The next cell writes `casmi_bench.py` next to this notebook. It holds the work being timed, so
that all four implementations run exactly the same Python functions:

- `clean_row_group` is the **map**. It reads one row group in batches and applies notebook 01's
  cleaning: the label rules, the copy keys, the peak rules and the 0.1 Da binning. It writes the
  cleaned row group to disk and returns counts and the row group's copy keys. Copies *inside* the
  row group are removed here, which is the MapReduce idea of a *combiner*: a reduce done locally
  before anything is sent anywhere.
- `find_copies` is the **reduce**. A spectrum can be a copy of one in an earlier row group, which
  no single task can see, so the copy keys of all row groups are brought together and only the
  first of each is kept. This is the one step that needs all of the data at once.
- `run_map` runs the map four ways. *One process* runs the tasks one after another with one
  thread. *Polars threads* does the same, but lets Polars use several threads inside the one
  process; the threads share memory. *joblib* starts several Python processes and hands each a
  task in turn, as in the Block 09 workshop. *Spark* starts a JVM in local mode, makes an RDD of
  row-group numbers and maps the same function over it, which is the Block 11 pattern.
- `load_trial` is the loading experiment of section 3.

The code is in a file rather than a notebook cell for a practical reason that Block 09 points out.
joblib's and Spark's workers are new Python processes on macOS and Windows (and joblib's are on
Linux too). They find a function by importing its module, and a function defined in a notebook
cell has no module they can import.

The map deliberately does notebook 01's work without the validation roles, which are a join on a
small table. Section 5.3 checks that its counts match notebook 01's.

In [ ]:
%%writefile casmi_bench.py
"""Benchmark workloads for notebook 02 of the data and validation strand (CASMI26).

The notebook writes this file next to itself and imports it. Worker processes (joblib's and
Spark's) start a fresh Python and import their functions by module name, so a function defined
in a notebook cell cannot be sent to them on macOS or Windows. Keeping the work in a module avoids
that, as in the Block 09 multiprocessing workshop.

Every benchmark trial runs in its own Python process (see `main`), so each starts from the same
state, sets its own thread count, and has its memory measured on its own.
"""
from __future__ import annotations

import json
import os
import re
import shutil
import sys
import tempfile
import threading
import time
from functools import lru_cache, partial

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

# Settings shared with notebook 01, section 5. They are passed in by the notebook; these are the defaults.
DEFAULTS = {
    "MIN_REL_INTENSITY": 0.005,
    "MAX_PEAKS": 256,
    "PRECURSOR_WINDOW": 2.5,
    "PRECURSOR_MZ_RANGE": (50.0, 1500.0),
    "MIN_PEAKS_TRAIN": 3,
    "BIN_WIDTH": 0.1,
    "BATCH_ROWS": 32_768,
}
CLEAN_COLUMNS = ["inchikey14", "adduct", "ionization_mode", "precursor_mz", "num_peaks",
                 "ms2_mzs", "ms2_normalized_intensities"]
COPY_KEY = ["inchikey14", "adduct", "num_peaks", "mz_sum", "intensity_sum"]


# ---------------------------------------------------------------------------------------------
# Adduct charge, as in notebook 01 section 3 (only the charge is needed for the cleaning rules).
# ---------------------------------------------------------------------------------------------
ADDUCT = re.compile(r"\[(\d*)M((?:[+-]\d*[A-Za-z][A-Za-z0-9]*)*)\](\d*)([+-])")
ADDUCT_TERM = re.compile(r"([+-])(\d*)([A-Za-z][A-Za-z0-9]*)")
FORMULA = re.compile(r"(?:[A-Z][a-z]?\d*)+")
ELEMENT_COUNT = re.compile(r"([A-Z][a-z]?)(\d*)")
ELEMENTS = {"H", "C", "N", "O", "S", "P", "F", "Cl", "Br", "I", "B", "Si", "As", "Se", "Sn",
            "Li", "Na", "K", "Mg", "Ca", "Fe", "Cu", "Zn", "Ag"}


@lru_cache(maxsize=None)
def adduct_charge(adduct: str | None) -> int | None:
    """The charge of an adduct such as '[M+H]+' (+1) or '[M-2H]2-' (-2); None if it cannot be parsed."""
    if adduct is None:
        return None
    match = ADDUCT.fullmatch(adduct)
    if match is None:
        return None
    for _, _, fragment in ADDUCT_TERM.findall(match.group(2)):
        if not FORMULA.fullmatch(fragment) or any(e not in ELEMENTS for e, _ in ELEMENT_COUNT.findall(fragment)):
            return None
    return int(match.group(3) or 1) * (1 if match.group(4) == "+" else -1)


# ---------------------------------------------------------------------------------------------
# The map: clean one row group. This is the per-spectrum work of notebook 01, sections 5 and 8.
# ---------------------------------------------------------------------------------------------
def clean_peaks(spectra, cfg):
    """Notebook 01's clean_peaks: noise floor, precursor window, peak cap, rescale, 0.1 Da bins."""
    import polars as pl

    max_fragment = cfg["PRECURSOR_MZ_RANGE"][1] + cfg["PRECURSOR_WINDOW"]
    kept = (
        spectra.lazy()
        .select(
            "row_id",
            (pl.col("precursor_mz") * pl.col("charge").abs() + cfg["PRECURSOR_WINDOW"])
            .clip(upper_bound=max_fragment).alias("mz_limit"),
            pl.col("ms2_mzs").alias("mz"),
            pl.col("ms2_normalized_intensities").alias("intensity"),
        )
        .explode("mz", "intensity")
        .filter((pl.col("intensity") >= cfg["MIN_REL_INTENSITY"]) & (pl.col("mz") <= pl.col("mz_limit")))
        .filter(pl.col("intensity").rank("ordinal", descending=True).over("row_id") <= cfg["MAX_PEAKS"])
        .with_columns(pl.col("intensity") / pl.col("intensity").max().over("row_id"))
    )
    peak_lists = kept.group_by("row_id", maintain_order=True).agg(
        pl.col("mz").alias("mzs"), pl.col("intensity").cast(pl.Float32).alias("intensities"))
    binned = (
        kept.with_columns((pl.col("mz") / cfg["BIN_WIDTH"] + 1e-9).floor().cast(pl.Int32).alias("bin"))
        .group_by("row_id", "bin").agg(pl.col("intensity").sum())
        .sort("row_id", "bin")
        .with_columns(pl.col("intensity") / pl.col("intensity").max().over("row_id"))
        .group_by("row_id", maintain_order=True)
        .agg(pl.col("bin").alias("bins"), pl.col("intensity").cast(pl.Float32).alias("bin_intensities"))
    )
    return (
        peak_lists.join(binned, on="row_id")
        .with_columns(pl.col("mzs").list.len().cast(pl.Int32).alias("n_peaks"))
        .collect()
    )


def clean_row_group(path: str, index: int, cfg: dict, out_dir: str | None = None) -> dict:
    """Clean one row group of train.parquet, a batch at a time, and write the result.

    Returns counts, the time taken, and the copy keys this row group keeps. Copies inside the row
    group are removed here (a local reduce, or "combiner"); copies of a spectrum in an *earlier* row
    group can only be found once every row group's keys are brought together (`find_copies`).
    """
    import polars as pl

    started = time.perf_counter()
    threads = int(os.environ.get("BENCH_THREADS", "0"))
    if threads:
        pa.set_cpu_count(threads)        # Arrow's decoding threads, as POLARS_MAX_THREADS sets Polars'
    cfg = {**DEFAULTS, **cfg}
    parquet = pq.ParquetFile(path, memory_map=False)
    start_row = sum(parquet.metadata.row_group(i).num_rows for i in range(index))
    low, high = cfg["PRECURSOR_MZ_RANGE"]
    counts = dict.fromkeys(["spectra in", "peaks in", "dropped: label rules", "dropped: copy within row group",
                            "dropped: no peak left", f"under {cfg['MIN_PEAKS_TRAIN']} peaks", "spectra out",
                            "peaks out"], 0)
    keys, parts, offset = [], [], start_row
    for batch in parquet.iter_batches(batch_size=cfg["BATCH_ROWS"], row_groups=[index], columns=CLEAN_COLUMNS):
        raw = pl.from_arrow(batch).with_row_index("row_id", offset=offset)
        offset += raw.height
        counts["spectra in"] += raw.height
        counts["peaks in"] += int(raw["num_peaks"].sum())

        # 1. The label rules, which need no peaks.
        charges = {a: adduct_charge(a) for a in raw["adduct"].unique().to_list()}
        raw = raw.with_columns(pl.col("adduct").replace_strict(charges, default=None, return_dtype=pl.Int8).alias("charge"))
        ok = (pl.col("charge").is_not_null()
              & ((pl.col("ionization_mode") == "positive") == (pl.col("charge") > 0))
              & pl.col("precursor_mz").is_between(low, high)).fill_null(False)
        usable = raw.filter(ok)
        counts["dropped: label rules"] += raw.height - usable.height

        # 2. Copy keys, as in notebook 01 section 5.4.
        usable = usable.with_columns(
            pl.col("ms2_mzs").list.sum().round(3).alias("mz_sum"),
            pl.col("ms2_normalized_intensities").list.sum().round(4).alias("intensity_sum"),
        ).with_columns(pl.struct(COPY_KEY).hash(seed=0).alias("copy_key"))
        keys.append(usable.select("row_id", "copy_key"))

        # 3. The peak rules and binning.
        parts.append(clean_peaks(usable.select("row_id", "precursor_mz", "charge", "ms2_mzs",
                                               "ms2_normalized_intensities"), cfg))

    keys = pl.concat(keys) if keys else pl.DataFrame(schema={"row_id": pl.UInt32, "copy_key": pl.UInt64})
    first = keys.filter(pl.col("copy_key").is_first_distinct())
    counts["dropped: copy within row group"] = keys.height - first.height
    cleaned = pl.concat(parts).join(first.select("row_id"), on="row_id", how="semi")
    counts["dropped: no peak left"] = first.height - cleaned.height
    counts[f"under {cfg['MIN_PEAKS_TRAIN']} peaks"] = int((cleaned["n_peaks"] < cfg["MIN_PEAKS_TRAIN"]).sum())
    counts["spectra out"] = cleaned.height
    counts["peaks out"] = int(cleaned["n_peaks"].sum())
    if out_dir is not None:
        cleaned.join(first, on="row_id").write_parquet(os.path.join(out_dir, f"part-{index:05d}.parquet"),
                                                       compression="zstd")
    return {"index": index, "counts": counts, "keys": first["copy_key"].to_numpy(),
            "seconds": time.perf_counter() - started, "pid": os.getpid()}


# ---------------------------------------------------------------------------------------------
# The reduce: copies of a spectrum held in an earlier row group.
# ---------------------------------------------------------------------------------------------
def find_copies(results: list[dict]) -> tuple[int, int]:
    """Bring every row group's copy keys together and keep the first in file order.

    Returns (copies across row groups, distinct spectra). This is the one step that needs all of
    the data at once: on a cluster it is a shuffle, with each key sent to the machine that owns it.
    """
    results = sorted(results, key=lambda r: r["index"])
    if not results:
        return 0, 0
    keys = np.concatenate([r["keys"] for r in results])
    distinct = np.unique(keys).size
    return int(keys.size - distinct), int(distinct)


# ---------------------------------------------------------------------------------------------
# Four ways to run the map over the row groups.
# ---------------------------------------------------------------------------------------------
def run_map(implementation: str, path: str, tasks: list[int], workers: int, cfg: dict, out_dir: str | None):
    """Run clean_row_group over `tasks`. Returns (results, seconds to start up, seconds for the map)."""
    work = partial(clean_row_group, path, cfg=cfg, out_dir=out_dir)
    started = time.perf_counter()
    if implementation in ("one process", "polars threads"):
        # One Python process. With "one process" Polars has one thread; with "polars threads" it
        # has `workers` threads that share the process's memory (set by POLARS_MAX_THREADS).
        ready = time.perf_counter()
        results = [work(task) for task in tasks]
    elif implementation == "joblib":
        from joblib import Parallel, delayed

        with Parallel(n_jobs=workers, backend="loky") as parallel:
            parallel(delayed(os.getpid)() for _ in range(workers))   # start the worker processes
            ready = time.perf_counter()
            results = parallel(delayed(work)(task) for task in tasks)
    elif implementation == "spark":
        from pyspark.sql import SparkSession

        spark = (
            SparkSession.builder.master(f"local[{workers}]").appName("casmi26-benchmark")
            .config("spark.ui.enabled", "false").config("spark.ui.showConsoleProgress", "false")
            .config("spark.driver.memory", cfg.get("SPARK_DRIVER_MEMORY", "2g"))
            .config("spark.python.worker.reuse", "true")
            .getOrCreate()
        )
        spark.sparkContext.setLogLevel("ERROR")
        context = spark.sparkContext
        context.parallelize(range(workers), workers).map(lambda x: os.getpid()).collect()   # start the Python workers
        ready = time.perf_counter()
        # One Spark task per row group: an RDD of row-group numbers, mapped and collected.
        results = context.parallelize(tasks, len(tasks)).map(work).collect()
        spark.stop()
    else:
        raise ValueError(implementation)
    finished = time.perf_counter()
    return results, ready - started, finished - ready


def run_trial(spec: dict) -> dict:
    """One timed run: the map over the chosen row groups, then the reduce."""
    out_dir = tempfile.mkdtemp(prefix="casmi26-bench-", dir=spec.get("scratch")) if spec.get("write", True) else None
    try:
        tasks = list(range(spec["row_groups"]))
        results, startup, mapping = run_map(spec["implementation"], spec["path"], tasks, spec["workers"],
                                            spec.get("cfg", {}), out_dir)
        started = time.perf_counter()
        across, distinct = find_copies(results)
        reduce = time.perf_counter() - started
    finally:
        if out_dir:
            shutil.rmtree(out_dir, ignore_errors=True)
    counts = {}
    for r in results:
        for k, v in r["counts"].items():
            counts[k] = counts.get(k, 0) + v
    counts["dropped: copy across row groups"] = across
    task_seconds = [r["seconds"] for r in sorted(results, key=lambda r: r["index"])]
    return {"startup_s": startup, "map_s": mapping, "reduce_s": reduce, "total_s": startup + mapping + reduce,
            "task_s": [float(t) for t in task_seconds],
            "processes_used": len({r["pid"] for r in results}), "counts": counts}


# ---------------------------------------------------------------------------------------------
# Loading: four ways to read the same columns.
# ---------------------------------------------------------------------------------------------
def load_trial(spec: dict) -> dict:
    """Read the first `row_groups` row groups' chosen columns, then compute a tiny summary.

    Eager methods hold the whole selection in memory; streaming methods hold one batch at a time.
    """
    import polars as pl

    path, columns, method = spec["path"], spec["columns"], spec["method"]
    parquet = pq.ParquetFile(path, memory_map=False)
    groups = list(range(spec["row_groups"]))
    n_rows = sum(parquet.metadata.row_group(i).num_rows for i in groups)
    # The summary touches every selected column, so no method can skip reading one: the number of
    # list elements in each list column, the number of characters in each text column, and the sum
    # of each numeric column. All four methods must agree on it.
    schema = parquet.schema_arrow
    kinds = {c: ("list" if str(schema.field(c).type).startswith(("list", "large_list")) else
                 "text" if str(schema.field(c).type) in ("string", "large_string") else "number") for c in columns}
    started = time.perf_counter()
    if method in ("polars", "polars streaming"):
        summary = [pl.len().alias("rows")] + [
            (pl.col(c).list.len().sum() if k == "list" else pl.col(c).str.len_chars().sum() if k == "text"
             else pl.col(c).cast(pl.Float64).sum()).alias(c) for c, k in kinds.items()]
        query = pl.scan_parquet(path).select(columns).head(n_rows)
        if method == "polars":
            frame = query.collect()                                   # the whole selection in memory
            row = frame.select(summary).row(0, named=True)
        else:
            row = query.select(summary).collect(engine="streaming").row(0, named=True)
    elif method == "pandas":
        # What pandas.read_parquet does with its default engine: read an Arrow table, convert it.
        frame = parquet.read_row_groups(groups, columns=columns).to_pandas()
        row = {"rows": len(frame)}
        for c, k in kinds.items():
            if k == "list":
                row[c] = frame[c].map(len, na_action="ignore").sum()
            elif k == "text":
                row[c] = frame[c].str.len().sum()
            else:
                row[c] = frame[c].astype("float64").sum()
    elif method == "pyarrow batches":
        import pyarrow.compute as pc

        row = dict.fromkeys(["rows", *columns], 0)
        # One row group at a time: given several row groups at once, pyarrow reads ahead and can
        # hold most of the selection in memory.
        batches = (batch for g in groups for batch in
                   parquet.iter_batches(batch_size=spec.get("batch_rows", 32_768), row_groups=[g], columns=columns))
        for batch in batches:
            row["rows"] += batch.num_rows
            for c, k in kinds.items():
                values = batch.column(c)
                part = (pc.sum(pc.list_value_length(values)) if k == "list" else
                        pc.sum(pc.utf8_length(values)) if k == "text" else pc.sum(pc.cast(values, "float64")))
                row[c] += part.as_py() or 0
    else:
        raise ValueError(method)
    seconds = time.perf_counter() - started
    assert row["rows"] == n_rows, (row["rows"], n_rows)
    return {"total_s": seconds, "rows": int(row["rows"]),
            "check": {c: round(float(v), 3) for c, v in row.items() if v is not None}}


# ---------------------------------------------------------------------------------------------
# Measurement: peak memory of this process and all its children (workers, Spark's JVM).
# ---------------------------------------------------------------------------------------------
class PeakMemory:
    """Sample the resident memory of this process tree every `interval` seconds; keep the peak.

    Resident memory counts pages shared between processes once per process, so for several
    workers this is an upper bound.
    """

    def __init__(self, interval: float = 0.05):
        import psutil

        self.process, self.interval, self.peak = psutil.Process(), interval, 0
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _sample(self):
        total = 0
        for p in [self.process, *self.process.children(recursive=True)]:
            try:
                total += p.memory_info().rss
            except Exception:
                pass
        self.peak = max(self.peak, total)

    def _run(self):
        while not self._stop.is_set():
            self._sample()
            time.sleep(self.interval)

    def __enter__(self):
        self._sample()
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join()
        self._sample()


def main():
    spec = json.loads(sys.argv[1])
    with PeakMemory() as memory:
        result = load_trial(spec) if spec["kind"] == "load" else run_trial(spec)
    result["peak_memory_mb"] = memory.peak / 1e6
    print("RESULT " + json.dumps(result))


if __name__ == "__main__":
    main()

### 2.2 How each run is timed

Timings on a laptop are noisy, so every run follows the same protocol:

- **A fresh process per run.** Each run starts its own Python process, which sets its thread
  count before Polars and Arrow are imported, and is measured on its own. Nothing left in memory
  by one run can speed up or slow down the next.
- **Thread counts are pinned.** `POLARS_MAX_THREADS` and Arrow's thread count are set to 1
  for the one-process, joblib and Spark runs, so that *p* workers use *p* cores. Without this,
  every joblib worker would start its own pool of threads, one per core, and they would compete.
- **Warm file cache.** The file is read once before any timing, so every run reads it from the
  operating system's cache rather than the disk. Cold-cache timings would need the cache to be
  dropped between runs, which needs administrator rights. The loading times are therefore an
  upper bound on the read speed of a real disk.
- **Repeats, interleaved.** Every configuration runs `REPEATS` times and the median is reported.
  The repeats are interleaved (all configurations once, then all again), so slow drift such as
  the laptop heating up affects every configuration alike.
- **Phases timed separately.** A cleaning run records the start-up of its workers, the map and
  the reduce, so the analysis can tell fixed costs from costs that grow with the data.
- **Memory.** A background thread samples the resident memory of the run's whole process tree
  (the run, its worker processes and Spark's JVM) every 0.05 s and keeps the peak. Memory shared
  between processes is counted once per process, so for several workers it is an upper bound.

The results are written to `benchmark_results/` after every run, so an interrupted session loses
nothing and the analysis can be re-run without the benchmarks (`RUN_BENCHMARKS = False`).

In [ ]:
LOAD_LOG, CLEAN_LOG = RESULTS_DIR / "loading.jsonl", RESULTS_DIR / "cleaning.jsonl"


def run_in_process(spec: dict, threads: int, timeout: int = 3600) -> dict:
    """Run one benchmark in a new Python process and return what it measured."""
    command = [sys.executable, "-c", "import casmi_bench; casmi_bench.main()", json.dumps(spec)]
    done = subprocess.run(command, env=worker_env(threads), cwd=HERE, capture_output=True, text=True, timeout=timeout)
    lines = [line for line in done.stdout.splitlines() if line.startswith("RESULT ")]
    if done.returncode != 0 or not lines:
        raise RuntimeError(f"benchmark failed: {spec}\n" + "\n".join(done.stderr.strip().splitlines()[-12:]))
    return json.loads(lines[-1][len("RESULT "):])


def record(log: Path, row: dict) -> None:
    with log.open("a") as handle:
        handle.write(json.dumps(row) + "\n")


def read_log(log: Path) -> list[dict]:
    return [json.loads(line) for line in log.read_text().splitlines()] if log.exists() else []


if RUN_BENCHMARKS:
    for log in (LOAD_LOG, CLEAN_LOG):
        log.unlink(missing_ok=True)
    (RESULTS_DIR / "machine.json").write_text(json.dumps({**machine, "java": JAVA_MAJOR, "spark": RUN_SPARK,
                                                          "quick": QUICK, "row_groups": FULL}, indent=2))
    # Warm the file cache: read every byte of the row groups the experiments use, untimed.
    started = time.perf_counter()
    warm = pq.ParquetFile(TRAIN)
    for index in range(FULL):
        warm.read_row_group(index)
    del warm
    print(f"read the data once to warm the cache, in {time.perf_counter() - started:.1f} s")
else:
    print("RUN_BENCHMARKS is off: analysing the saved results in", RESULTS_DIR)

## 3. Loading

Four ways of reading the same columns of the first *k* row groups, at each data size in
`SIZE_STEPS`, for two choices of columns: everything, and the metadata only (every column but the
peak lists and collision energies):

| Method | What it does | What it holds in memory |
| --- | --- | --- |
| pandas | Reads an Arrow table and converts it to a pandas DataFrame, which is what `pd.read_parquet` does. Each peak list becomes a Python object. | The whole selection |
| polars | `pl.scan_parquet(...).select(...).collect()`: Polars' own reader, in Arrow's memory layout. | The whole selection |
| polars streaming | The same query run by Polars' streaming engine, which reads, summarises and discards one chunk at a time. | One chunk |
| pyarrow batches | A loop over row groups and batches of 32,768 rows. | One batch |

So that no method can skip a column it was asked for, each computes the same summary of
everything it reads: the number of rows, the list elements in each list column, the characters
in each text column and the sum of each numeric column. The four summaries must agree.

The two in-memory methods are not run where their memory is predicted to exceed
`MEMORY_BUDGET_GB`, from the uncompressed size of the selection; skipped runs are listed.

In [ ]:
column_bytes = {}   # uncompressed bytes of each column in each row group
for g in range(N_ROW_GROUPS):
    group = layout.row_group(g)
    for j in range(group.num_columns):
        name = group.column(j).path_in_schema.split(".")[0]
        column_bytes.setdefault(name, np.zeros(N_ROW_GROUPS))[g] += group.column(j).total_uncompressed_size

COLUMN_SETS = {"all columns": ALL_COLUMNS, "metadata only": METADATA_COLUMNS}
MEMORY_FACTOR = {"pandas": 4.0, "polars": 1.5}   # rough peak memory per uncompressed byte, for the skip rule


def selection_bytes(columns: list[str], row_groups: int) -> float:
    return float(sum(column_bytes[c][:row_groups].sum() for c in columns))


if RUN_BENCHMARKS:
    skipped = []
    for repeat in range(REPEATS):
        for size in SIZES:
            for column_set, columns in COLUMN_SETS.items():
                for method in LOADERS:
                    predicted_gb = selection_bytes(columns, size) * MEMORY_FACTOR.get(method, 0) / 1e9
                    if predicted_gb > MEMORY_BUDGET_GB:
                        if repeat == 0:
                            skipped.append(f"{method}, {column_set}, {size} row groups (about {predicted_gb:.0f} GB)")
                        continue
                    spec = {"kind": "load", "method": method, "path": str(TRAIN), "row_groups": size, "columns": columns}
                    result = run_in_process(spec, threads=LOGICAL_CORES)
                    record(LOAD_LOG, {"repeat": repeat, "method": method, "columns": column_set, "row_groups": size,
                                      "spectra": int(ROW_GROUP_ROWS[:size].sum()),
                                      "bytes": selection_bytes(columns, size), **result})
        print(f"loading, repeat {repeat + 1} of {REPEATS} done")
    print("skipped:", *skipped, sep="\n  ") if skipped else print("nothing skipped")

loading_runs = read_log(LOAD_LOG)
loading = (
    pl.DataFrame([{k: v for k, v in r.items() if k != "check"} for r in loading_runs])
    .group_by("method", "columns", "row_groups", "spectra", "bytes")
    .agg(pl.col("total_s").median().alias("seconds"), pl.col("peak_memory_mb").median().alias("peak MB"))
    .with_columns((pl.col("bytes") / 1e6 / pl.col("seconds")).alias("MB per s"))
    .sort("columns", "method", "row_groups")
)

# The four methods must report the same summary for the same selection.
agree = True
for (column_set, size), runs in pl.DataFrame(
        [{"columns": r["columns"], "row_groups": r["row_groups"], "i": i} for i, r in enumerate(loading_runs)]
).group_by("columns", "row_groups"):
    checks = [loading_runs[i]["check"] for i in runs["i"]]
    for other in checks[1:]:
        agree &= all(np.isclose(other[k], checks[0][k], rtol=1e-9, atol=1e-6) for k in checks[0])
print("all methods read the same values:", agree)
loading.filter(pl.col("row_groups") == pl.col("row_groups").max()).drop("bytes")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6.4), sharex=True)
for row, column_set in enumerate(COLUMN_SETS):
    for method, colour in LOADERS.items():
        part = loading.filter((pl.col("method") == method) & (pl.col("columns") == column_set))
        if part.height:
            axes[row, 0].plot(part["spectra"], part["seconds"], "o-", color=colour, label=method, markersize=4)
            axes[row, 1].plot(part["spectra"], part["peak MB"], "o-", color=colour, label=method, markersize=4)
    axes[row, 0].set_ylabel(f"{column_set}\nseconds")
    axes[row, 1].set_ylabel("peak memory (MB)")
for ax in axes.flat:
    ax.set_xscale("log")
    ax.set_yscale("log")
for ax in axes[1]:
    ax.set_xlabel("spectra read (log scale)")
axes[0, 0].set_title("Time to read")
axes[0, 1].set_title("Peak memory while reading")
axes[0, 0].legend(fontsize=8)
fig.suptitle("In-memory reads grow with the data; streaming reads hold one batch at a time",
             x=0.01, ha="left", fontweight="bold", color=INK)
fig.tight_layout()
plt.show()

In [ ]:
# Slopes of the log-log lines: memory ~ spectra^slope. A slope near 1 means memory grows in
# proportion to the data; near 0 means it does not grow.
slopes = []
for (method, column_set), part in loading.group_by("method", "columns"):
    if part.height >= 3:
        x, y = np.log(part["spectra"].to_numpy()), np.log(part["peak MB"].to_numpy())
        t = np.log(part["seconds"].to_numpy())
        slopes.append({"method": method, "columns": column_set,
                       "time ~ spectra^": np.polyfit(x, t, 1)[0], "memory ~ spectra^": np.polyfit(x, y, 1)[0]})
pl.DataFrame(slopes).sort("columns", "method") if slopes else print("too few data sizes to fit slopes")

**Reading the figure.** The left panels show time and the right panels peak memory, both on log
scales, so a straight line of slope 1 means growth in proportion to the data. The table gives
those slopes.

- Reading the metadata only is far cheaper than reading everything, because Parquet stores
  columns separately and the peak lists are almost all of the file (notebook 01, section 2.1).
  That is the first lever at scale: read only the columns a step needs.
- pandas is the slowest and the most memory-hungry for the peak lists, because it turns each
  list into a separate Python object. Polars keeps Arrow's compact columnar layout.
- The two streaming methods hold one batch at a time, so their memory stays roughly flat
  while the data grows. The in-memory methods grow with the data, and at 1000 times the data
  (3 TB) they could not run on any single machine. This is why the cleaning pass streams: its
  memory is set by the batch size, not the file size.

## 4. The cleaning runs

One cell runs every cleaning benchmark, so the repeats can be interleaved. There are two
experiments:

- **Data volume** (section 5): one process with one thread, on the first *k* row groups for each
  *k* in `SIZE_STEPS`.
- **Cores** (section 6): every implementation at every worker count in `WORKER_COUNTS`, on all
  the row groups (4 with `QUICK`). For *Polars threads* the worker count is the number of threads
  in the one process; for joblib and Spark it is the number of worker processes, each with one
  thread.

In [ ]:
IMPLEMENTATIONS_RUN = [i for i in IMPLEMENTATIONS if i != "spark" or RUN_SPARK]
CONFIG = {**CLEANING, "SPARK_DRIVER_MEMORY": SPARK_DRIVER_MEMORY}


def cleaning_spec(implementation: str, workers: int, row_groups: int) -> dict:
    return {"kind": "clean", "implementation": implementation, "workers": workers, "row_groups": row_groups,
            "path": str(TRAIN), "scratch": str(SCRATCH), "cfg": CONFIG}


plan = []   # (experiment, implementation, workers, row groups), in the order of one repeat
plan += [("data volume", "one process", 1, size) for size in SIZES]
plan += [("cores", implementation, p, FULL) for p in WORKER_COUNTS for implementation in IMPLEMENTATIONS_RUN
         if implementation != "one process" and not (implementation == "polars threads" and p == 1)]

if RUN_BENCHMARKS:
    SCRATCH.mkdir(parents=True, exist_ok=True)
    spark_failures = 0
    print(f"{len(plan)} runs per repeat, {REPEATS} repeats")
    for repeat in range(REPEATS):
        for experiment, implementation, workers, size in plan:
            if implementation == "spark" and spark_failures >= 2:
                continue
            threads = workers if implementation == "polars threads" else 1
            try:
                result = run_in_process(cleaning_spec(implementation, workers, size), threads=threads)
            except (RuntimeError, subprocess.TimeoutExpired) as error:
                if implementation != "spark":
                    raise
                spark_failures += 1
                print(f"Spark run failed ({spark_failures} of 2 allowed):", str(error)[-600:])
                continue
            record(CLEAN_LOG, {"repeat": repeat, "experiment": experiment, "implementation": implementation,
                               "workers": workers, "row_groups": size, **result})
            print(f"repeat {repeat + 1}: {implementation:>14}, {workers} worker{'s' if workers > 1 else ' '}, "
                  f"{size:>2} row groups: {result['total_s']:6.1f} s, peak {result['peak_memory_mb']:6,.0f} MB")
    shutil.rmtree(SCRATCH, ignore_errors=True)

cleaning_runs = read_log(CLEAN_LOG)
runs = pl.DataFrame([
    {k: v for k, v in r.items() if k not in ("counts", "task_s")} | {"spectra": r["counts"]["spectra in"]}
    for r in cleaning_runs
])
summary = (
    runs.group_by("experiment", "implementation", "workers", "row_groups", "spectra")
    .agg(pl.col("startup_s", "map_s", "reduce_s", "total_s", "peak_memory_mb").median(), pl.len().alias("repeats"),
         (pl.col("total_s").max() - pl.col("total_s").min()).alias("range_s"))
    .sort("experiment", "implementation", "workers", "row_groups")
)
print(f"{runs.height} cleaning runs loaded")

### 4.1 Do all the implementations compute the same thing?

A benchmark of a wrong answer is worthless, so before comparing speeds the notebook checks the
counts. Every run over the same row groups must report the same numbers of spectra and peaks in,
out and dropped at each rule, whatever the implementation and worker count. When all row groups
are used, the label-rule and copy counts must also equal notebook 01's (its `manifest.json`). The
other counts differ from notebook 01's by design, because this pass has no held-out molecules.

In [ ]:
def counts_of(r: dict) -> tuple:
    return tuple(sorted(r["counts"].items()))


full_counts = {counts_of(r) for r in cleaning_runs if r["row_groups"] == FULL}
assert len(full_counts) == 1, "runs over the same data disagree"
counts = dict(next(iter(full_counts)))
print(f"all {sum(r['row_groups'] == FULL for r in cleaning_runs)} runs over {FULL} row groups report identical counts")

if MANIFEST.exists() and FULL == N_ROW_GROUPS:
    reference = json.loads(MANIFEST.read_text())
    if reference["settings"]["SUBSAMPLE"] == 1 and reference["source"]["rows"] == layout.num_rows:
        copies = counts["dropped: copy within row group"] + counts["dropped: copy across row groups"]
        assert counts["dropped: label rules"] == reference["counts"]["dropped: label rules"]
        assert copies == reference["counts"]["dropped: exact copy"]
        print(f"label rules ({counts['dropped: label rules']:,}) and exact copies ({copies:,}) match notebook 01")
    else:
        print("notebook 01 was run on a subsample, so its counts are not comparable")
else:
    print("notebook 01's manifest not found, or only part of the file used: comparison skipped")
pl.DataFrame({"count": list(counts), "value": list(counts.values())})

## 5. Data volume: does the cost grow linearly?

The 1000× estimate in section 7 multiplies a measured cost by 1000. That is only right if the
cost is proportional to the amount of data. The figure shows the one-process time and peak
memory against the number of spectra, using nested subsets of the file (the first 1, 2, 5, 10
and all row groups).

In [ ]:
volume = summary.filter(pl.col("experiment") == "data volume").sort("spectra")
spectra = volume["spectra"].to_numpy().astype(float)
seconds = volume["total_s"].to_numpy()
per_spectrum, fixed = np.polyfit(spectra, seconds, 1)          # seconds = fixed + per_spectrum * spectra
fitted = fixed + per_spectrum * spectra
r_squared = 1 - np.sum((seconds - fitted) ** 2) / np.sum((seconds - seconds.mean()) ** 2) if len(spectra) > 2 else float("nan")

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.4))
left.plot(spectra / 1e6, seconds, "o", color=GREY, label="median of the repeats")
grid = np.linspace(0, spectra.max(), 50)
left.plot(grid / 1e6, fixed + per_spectrum * grid, color=INK_SOFT, linewidth=1, label="least-squares line")
left.set_xlabel("spectra cleaned (millions)")
left.set_ylabel("seconds, one core")
left.set_title("Time grows in proportion to the data")
left.legend(fontsize=8)
right.plot(spectra / 1e6, volume["peak_memory_mb"], "o-", color=GREY)
right.set_ylim(0, volume["peak_memory_mb"].max() * 1.3)
right.set_xlabel("spectra cleaned (millions)")
right.set_ylabel("peak memory (MB)")
right.set_title("Memory does not")
fig.tight_layout()
plt.show()
print(f"fit: {per_spectrum * 1e6:.1f} µs per spectrum ({1 / per_spectrum:,.0f} spectra per second on one core), "
      f"fixed cost {fixed:.2f} s, R² = {r_squared:.4f}")
print(f"peak memory ranges from {volume['peak_memory_mb'].min():,.0f} to {volume['peak_memory_mb'].max():,.0f} MB "
      f"while the data grows {spectra.max() / spectra.min():.0f}-fold")

The time is close to a straight line through the origin, so each spectrum costs about the same
whatever else is in the file, and a fixed cost (starting Python, importing the libraries) is
small. The memory barely moves, because the pass holds one batch of 32,768 spectra at a time and
writes each row group out before reading the next. Both are what a 1000× extrapolation needs: the
time scales by the factor, and the memory of one worker does not scale at all.

One caveat applies. The row groups are not identical: the libraries are not spread evenly
through the file, and spectra from different libraries have different numbers of peaks. So the
cost per spectrum of a subset can differ a little from the whole, which shows as scatter about
the line.

## 6. Cores: four implementations

### 6.1 Speed-up

The *speed-up* of a run on *p* cores is $S(p) = T_1 / T(p)$, where $T_1$ is the time of the
one-process, one-thread run over the same data. Perfect use of *p* cores gives $S(p) = p$, the
dashed diagonal. Every run is measured from the moment the work is asked for, so Spark's and
joblib's start-up counts against them, as it would in practice.

The grey line is a bound that comes from the tasks themselves. There are only as many tasks as
row groups, and they differ in length. Handing them out in order to whichever worker is free
next, which is what joblib and Spark do, the run cannot finish before the most-loaded worker
does. The bound is the speed-up of that schedule, computed from each row group's one-core time
with no overhead at all.

In [ ]:
baseline = volume.filter(pl.col("row_groups") == FULL).row(0, named=True)
T1, T1_MAP = baseline["total_s"], baseline["map_s"]
cores = pl.concat([
    summary.filter(pl.col("experiment") == "cores"),
    pl.DataFrame([baseline]).with_columns(pl.lit("cores").alias("experiment"), pl.lit("polars threads").alias("implementation")),
], how="vertical_relaxed").with_columns(
    (T1 / pl.col("total_s")).alias("speed-up"),
    (T1_MAP / pl.col("map_s")).alias("map speed-up"),
).with_columns((pl.col("speed-up") / pl.col("workers")).alias("efficiency")).sort("implementation", "workers")

# The schedule bound, from each row group's median one-core time.
task_times = np.median([r["task_s"] for r in cleaning_runs
                        if r["implementation"] == "one process" and r["row_groups"] == FULL], axis=0)


def in_order_makespan(times: np.ndarray, workers: int) -> float:
    """Finish time when tasks are handed out in order to whichever worker is free first."""
    free = np.zeros(workers)
    for t in times:
        free[np.argmin(free)] += t
    return free.max()


bound = {p: task_times.sum() / in_order_makespan(task_times, p) for p in WORKER_COUNTS}


def fit_amdahl(p: np.ndarray, seconds: np.ndarray) -> tuple[float, float]:
    """Least-squares fit of T(p) = serial + parallel / p (section 6.4). Returns (serial, parallel) in seconds."""
    x = 1 / p
    parallel = np.sum((x - x.mean()) * (seconds - seconds.mean())) / np.sum((x - x.mean()) ** 2)
    return float(seconds.mean() - parallel * x.mean()), float(parallel)


amdahl_fit, serial_fraction = {}, {}
fig, ax = plt.subplots(figsize=(8, 4.6))
top = max(WORKER_COUNTS)
ax.plot([1, top], [1, top], "--", color=MUTED, linewidth=1, label="perfect: S(p) = p")
ax.plot(list(bound), list(bound.values()), color=GRID, linewidth=6, solid_capstyle="round",
        label=f"best possible with {FULL} unequal tasks", zorder=0)
p_grid = np.linspace(1, top, 100)
for implementation in IMPLEMENTATIONS_RUN:
    if implementation == "one process":
        continue
    part = cores.filter(pl.col("implementation") == implementation)
    if not part.height:
        continue
    p, s = part["workers"].to_numpy().astype(float), part["speed-up"].to_numpy()
    serial, parallel = amdahl_fit[implementation] = fit_amdahl(p, part["total_s"].to_numpy())
    f = serial_fraction[implementation] = serial / (serial + parallel)
    colour = IMPLEMENTATIONS[implementation]
    ax.plot(p, s, "o-", color=colour, label=f"{implementation} (fitted f = {f:.2f})", markersize=5)
    if len(p) > 1:
        ax.plot(p_grid, T1 / (serial + parallel / p_grid), ":", color=colour, linewidth=1.2)
if PHYSICAL_CORES and PHYSICAL_CORES < top:
    ax.axvline(PHYSICAL_CORES, color=INK_SOFT, linewidth=0.8)
    ax.annotate(f"{PHYSICAL_CORES} physical cores", (PHYSICAL_CORES, 0.98), xycoords=("data", "axes fraction"),
                xytext=(4, -10), textcoords="offset points", fontsize=8, color=INK_SOFT)
ax.set_xlabel("workers: processes, or threads for Polars threads")
ax.set_ylabel("speed-up over one core")
ax.set_xticks(WORKER_COUNTS)
ax.set_title("Speed-up of the cleaning pass, with Amdahl's law fitted (dotted)")
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()
cores.select("implementation", "workers", "total_s", "speed-up", "efficiency", "map speed-up", "peak_memory_mb", "range_s")

### 6.2 Where the time goes

Each run's time splits into three phases: starting the workers (a pool of processes for joblib;
a JVM, a Spark context and Python workers for Spark), the map, and the reduce. The bars show the
split for one core and for the most workers tried.

In [ ]:
phase_rows = cores.filter(pl.col("workers").is_in([1, top])).sort("workers", "implementation")
labels = [f"{r['implementation']}\n{r['workers']} worker{'s' if r['workers'] > 1 else ''}"
          for r in phase_rows.iter_rows(named=True)]
fig, ax = plt.subplots(figsize=(9, 3.6))
left = np.zeros(phase_rows.height)
for phase, colour in [("startup_s", ORANGE), ("map_s", BLUE), ("reduce_s", INK_SOFT)]:
    values = phase_rows[phase].to_numpy()
    ax.barh(labels, values, left=left, color=colour, label=phase.replace("_s", ""), height=0.6)
    left += values
ax.invert_yaxis()
ax.set_xlabel("seconds (median of the repeats)")
ax.set_title("Start-up is a fixed cost; the map is what parallelises")
fig.legend(fontsize=8, ncols=3, loc="upper right")
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()
for r in phase_rows.filter(pl.col("workers") == top).iter_rows(named=True):
    print(f"{r['implementation']:>14}, {top} workers: start-up {r['startup_s']:5.1f} s, map {r['map_s']:6.1f} s, "
          f"reduce {r['reduce_s']:.2f} s")

### 6.3 Memory

Threads share one process's memory; processes each hold their own copy of the libraries and of
the batch they are working on. Spark adds a JVM.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
for implementation in IMPLEMENTATIONS_RUN:
    part = cores.filter(pl.col("implementation") == implementation)
    if part.height and implementation != "one process":
        ax.plot(part["workers"], part["peak_memory_mb"] / 1e3, "o-", color=IMPLEMENTATIONS[implementation],
                label=implementation, markersize=5)
ax.set_xlabel("workers")
ax.set_ylabel("peak memory (GB)")
ax.set_xticks(WORKER_COUNTS)
ax.set_ylim(bottom=0)
ax.set_title("Processes multiply the memory; threads do not")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 6.4 The mathematics: Amdahl's law, its fit and the Karp–Flatt metric

**Amdahl's law** [(Amdahl, 1967)](https://doi.org/10.1145/1465482.1465560) splits the time of a
job into a part that cannot be parallelised, $T_s$, and a part that divides perfectly over $p$
cores, $T_{par}$:

$$T(p) = T_s + \frac{T_{par}}{p}, \qquad f = \frac{T_s}{T_s + T_{par}}, \qquad S(p) = \frac{T(1)}{T(p)} = \frac{1}{f + (1-f)/p} \le \frac{1}{f}.$$

$f$ is the *serial fraction*. However many cores are added, the speed-up cannot pass $1/f$. In
our pass, $T_s$ collects everything that does not divide among the workers: starting them, the
reduce, collecting the results, and work done in the parent process.

**Fitting it.** $T(p)$ is linear in $x = 1/p$, so $T_s$ and $T_{par}$ are the intercept and slope
of an ordinary least-squares line through the points $(1/p, T(p))$:

$$\hat T_{par} = \frac{\sum_p (x_p - \bar x)(T_p - \bar T)}{\sum_p (x_p - \bar x)^2}, \qquad \hat T_s = \bar T - \hat T_{par}\,\bar x.$$

Each implementation gets its own fit, so Spark's start-up lands in its $T_s$ rather than being
mistaken for poor scaling. The dotted curves in section 6.1 are $T_1 / \hat T(p)$, the fitted
speed-up over the one-process run, and the legend gives $\hat f$. With only two or three worker
counts the fit is rough; the more counts, the better it is determined.

**The Karp–Flatt metric** [(Karp and Flatt, 1990)](https://doi.org/10.1145/78607.78614) inverts
Amdahl's law at each $p$ separately. With $S(p)$ the speed-up over one core,

$$e(p) = \frac{1/S(p) - 1/p}{1 - 1/p}$$

is the serial fraction that would explain that one measurement. If Amdahl's law were the whole
story, $e(p)$ would be the same at every $p$. If it rises with $p$, the overhead itself grows with
the number of workers (more processes to start, more results to collect, more competition for
memory bandwidth), which the law does not model. The table shows $e(p)$, measured against the
one-process run, so a value above 1 means the run was slower than one core.

In [ ]:
karp_flatt = (
    cores.filter(pl.col("workers") > 1)
    .with_columns(((1 / pl.col("speed-up") - 1 / pl.col("workers")) / (1 - 1 / pl.col("workers"))).alias("e(p)"))
    .pivot(on="workers", index="implementation", values="e(p)")
)
display(karp_flatt)
for implementation, (serial, parallel) in amdahl_fit.items():
    f = serial_fraction[implementation]
    limit = f"so it can never beat one core by more than {T1 / serial:.1f}×" if serial > 0 else "no serial part detected"
    print(f"{implementation:>14}: T_s = {serial:6.1f} s, T_par = {parallel:7.1f} s, f = {f:.3f}; {limit}")

**Two more limits** come from the data rather than the code:

- **Few, unequal tasks.** With $n$ tasks of equal length on $p$ workers, the last round of tasks
  runs on only $n - p\lfloor n/p \rfloor$ workers, so $S(p) \le n / \lceil n/p \rceil$. With 21
  row groups on 8 workers that is $21/3 = 7$, not 8. Unequal tasks lower the bound further. The
  remedy at scale is many more, smaller tasks than workers, which a 1000-fold larger file
  provides naturally: 21,000 row groups.
- **Physical cores.** Two logical cores on one physical core share its arithmetic units. A
  CPU-bound job like this one usually gains much less from the second logical core than from the
  first, so the speed-up tends to flatten past the number of physical cores.

**Amdahl and Gustafson.** Amdahl's law holds the problem size fixed. At 1000 times the data the
map grows 1000-fold (section 5) while the start-up does not, so the serial fraction shrinks and
more workers pay off. This is Gustafson's law
[(Gustafson, 1988)](https://doi.org/10.1145/42411.42415). The exception is the reduce: the number
of copy keys grows with the data, so section 7 treats it separately.

## 7. What 1000 times the data would take

The estimate combines the measurements with the assumptions in the settings cell, and every
figure in the table comes from one of the two:

| Quantity | Source |
| --- | --- |
| One-core cleaning cost per spectrum | Measured: the slope in section 5 |
| Speed-up of the map on this machine | Measured: the best implementation at the most workers, section 6.1 |
| Reduce cost per copy key | Measured: the one-core reduce time divided by the number of keys |
| Machines in a cluster, network speed, disk speed | Assumed: `CLUSTER_MACHINES`, `NETWORK_GBIT_PER_S`, `DISK_MB_PER_S` |

The map at scale is spread over 1000 times as many row groups, so the task-count limit of
section 6.4 disappears and the cluster is assumed to keep each machine's measured speed-up.
Reading from disk is shown separately because the measurements read from a warm cache.

In [ ]:
spectra_full = layout.num_rows
spectra_scaled = SCALE * spectra_full
best = cores.filter((pl.col("workers") == top) & (pl.col("implementation") != "one process")).sort("map speed-up").row(-1, named=True)
map_speedup = best["map speed-up"]

keys_per_spectrum = (counts["spectra in"] - counts["dropped: label rules"]) / counts["spectra in"]
keys_scaled = keys_per_spectrum * spectra_scaled
reduce_per_key = baseline["reduce_s"] / (keys_per_spectrum * baseline["spectra"])
keys_gb = keys_scaled * 8 / 1e9                                  # 64-bit keys
disk_bytes = SCALE * FILE_BYTES


def hours(seconds: float) -> float:
    return seconds / 3600


core_seconds = per_spectrum * spectra_scaled
scenarios = [
    ("one core", 1, 1, core_seconds, reduce_per_key * keys_scaled, 0.0),
    (f"this machine, {best['implementation']}, {top} workers", 1, top, core_seconds / map_speedup,
     reduce_per_key * keys_scaled, 0.0),
    (f"cluster of {CLUSTER_MACHINES} such machines", CLUSTER_MACHINES, top * CLUSTER_MACHINES,
     core_seconds / (map_speedup * CLUSTER_MACHINES), reduce_per_key * keys_scaled / CLUSTER_MACHINES,
     keys_gb * 1e9 * (CLUSTER_MACHINES - 1) / CLUSTER_MACHINES / (NETWORK_GBIT_PER_S * 1e9 / 8) / CLUSTER_MACHINES),
]
projection = pl.DataFrame([
    {"scenario": name, "cores": n_cores, "map (h)": hours(m), "reduce (h)": hours(r), "shuffle (h)": hours(sh),
     "read from disk (h)": hours(disk_bytes / (DISK_MB_PER_S * 1e6 * machines)),
     "total (h)": hours(m + r + sh)}
    for name, machines, n_cores, m, r, sh in scenarios
])
display(projection)
print(f"1000× the data: {spectra_scaled / 1e9:.1f} billion spectra, {disk_bytes / 1e12:.1f} TB of Parquet, "
      f"{keys_gb:.0f} GB of copy keys for the reduce")
print(f"best map speed-up here: {map_speedup:.1f}× with {best['implementation']} on {top} workers")
if keys_gb > MEMORY_GB:
    print(f"the copy keys ({keys_gb:.0f} GB) do not fit in this machine's {MEMORY_GB:.0f} GB of memory, so on one "
          f"machine the reduce needs an external sort; on a cluster each machine holds 1/{CLUSTER_MACHINES} of them")

**Reading the projection.**

- **The map is the cost, and it divides.** The map dominates, and the tasks are independent, so
  its time divides by the number of cores available, up to each machine's measured efficiency.
- **The reduce changes nature, not size.** At this file's size the copy keys are about 20 MB and
  the reduce takes a fraction of a second. At 1000 times, they are tens of GB and no longer fit in
  a laptop's memory. On a cluster the keys are partitioned by their own value, so each machine
  receives every copy of the keys it owns and can decide alone which one is first. That exchange
  is the *shuffle*, the one point where machines must talk, and its time is set by the network.
  It is Spark's `reduceByKey`, and the reason a system like Spark exists: joblib can spread the
  map over one machine's cores but has no shuffle between machines.
- **Reading is not free.** 3 TB from disk takes as long as, or longer than, the cleaning on one
  machine, which is why a cluster spreads the data itself over the machines' disks (HDFS, or
  object storage) and runs each task where its row group is stored. Moving the computation to the
  data, rather than the data to the computation, is the distributed-data paradigm of Block 11.
- **Memory does not grow.** Section 5 showed one worker's memory is set by the batch size. At
  1000 times, each worker still needs the same few hundred MB, so the same laptops would do; only
  their number changes.

## 8. Summary

The cell below prints the measured answer to each of the three questions.

In [ ]:
largest = loading.filter(pl.col("row_groups") == pl.col("row_groups").max())
for column_set in COLUMN_SETS:
    part = largest.filter(pl.col("columns") == column_set).sort("seconds")
    if part.height:
        fastest = part.row(0, named=True)
        print(f"Loading, {column_set}: fastest is {fastest['method']} ({fastest['seconds']:.2f} s, "
              f"{fastest['peak MB']:,.0f} MB) of {part.height} methods run on {fastest['row_groups']} row groups.")
print(f"Data volume: {1 / per_spectrum:,.0f} spectra per second on one core, linear (R² = {r_squared:.3f}), "
      f"with memory flat at about {volume['peak_memory_mb'].median():,.0f} MB.")
for implementation, f in serial_fraction.items():
    row = cores.filter((pl.col("implementation") == implementation) & (pl.col("workers") == top))
    if row.height:
        r = row.row(0, named=True)
        print(f"Cores, {implementation}: {r['speed-up']:.1f}× on {top} workers ({r['efficiency']:.0%} efficiency), "
              f"fitted serial fraction {f:.2f}, peak memory {r['peak_memory_mb'] / 1e3:.1f} GB.")
print(f"1000× the data: about {projection['total (h)'][0]:,.0f} hours on one core, "
      f"{projection['total (h)'][1]:,.0f} hours on this machine, {projection['total (h)'][2]:,.1f} hours on "
      f"{CLUSTER_MACHINES} machines, before reading from disk.")

**Limitations.**

- Spark ran in *local mode*: one JVM on one machine, with no network and no distributed file
  system. It measures Spark's overheads, not a cluster's. A cluster would add the shuffle cost
  that section 7 estimates from an assumed network speed.
- Spark ran our Python function on an RDD, the pattern of Block 11. Spark's own DataFrame engine
  could run the peak rules inside the JVM without Python workers, and would likely be faster; we
  did not implement that version.
- All reads came from a warm file cache, so they are faster than a disk would be.
- One laptop, one operating system. Speed-ups depend on the processor (physical cores, memory
  bandwidth) and would differ on another machine, which is why section 1.1 records it.

## References

- Amdahl, G. M. (1967). Validity of the single processor approach to achieving large scale
  computing capabilities. *AFIPS Conference Proceedings* 30, 483–485.
  [doi:10.1145/1465482.1465560](https://doi.org/10.1145/1465482.1465560)
- Gustafson, J. L. (1988). Reevaluating Amdahl's law. *Communications of the ACM* 31(5), 532–533.
  [doi:10.1145/42411.42415](https://doi.org/10.1145/42411.42415)
- Karp, A. H. and Flatt, H. P. (1990). Measuring parallel processor performance. *Communications
  of the ACM* 33(5), 539–543. [doi:10.1145/78607.78614](https://doi.org/10.1145/78607.78614)
- Dean, J. and Ghemawat, S. (2008). MapReduce: simplified data processing on large clusters.
  *Communications of the ACM* 51(1), 107–113.
  [doi:10.1145/1327452.1327492](https://doi.org/10.1145/1327452.1327492)
- Zaharia, M. et al. (2012). Resilient distributed datasets: a fault-tolerant abstraction for
  in-memory cluster computing. *NSDI 2012*.
  [usenix.org](https://www.usenix.org/conference/nsdi12/technical-sessions/presentation/zaharia)
- Apache Spark 4.3.0 documentation: [overview](https://spark.apache.org/docs/latest/) and
  [PySpark installation](https://spark.apache.org/docs/latest/api/python/getting_started/install.html).
- joblib documentation: [embarrassingly parallel for loops](https://joblib.readthedocs.io/en/stable/parallel.html).
- Lawson, D. Data Science Toolbox course notes, Block 09 (parallel algorithms; the map/reduce and
  multiprocessing workshops) and Block 11 (parallel infrastructure and Spark).
  [dsbristol.github.io/dst](https://dsbristol.github.io/dst/)